# tinydistill quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Andreas-Hum/tinydistill/blob/main/notebooks/quickstart.ipynb)

Turn an LLM classification prompt into a small, fast classifier, in about ten minutes.

[tinydistill](https://github.com/Andreas-Hum/tinydistill) has an instruction-tuned LLM (the *teacher*) score
every label for every training text, trains a small encoder (the *student*) to imitate those soft labels, and
exports it to ONNX for fast CPU inference. No gold labels are needed.

This notebook distils the sentiment of financial news tweets (bearish, bullish, neutral) from
`Qwen/Qwen2.5-1.5B-Instruct` into `all-MiniLM-L6-v2` on 2,000 tweets, then shows what happened with charts.
**Use a GPU runtime** (Runtime > Change runtime type > T4 GPU); on a CPU the teacher is much slower.

In [ ]:
!pip install -q "tinydistill[hf,plot]"

Downloads from the Hugging Face Hub work without an account. If they are slow, or you see a warning about
unauthenticated requests, add a token: in Colab, open the key icon in the left sidebar and add a secret named
`HF_TOKEN` with a token from https://huggingface.co/settings/tokens (read access is enough).

## 1. Describe the task

A task is a short YAML file: the prompt, with `{text}` where the input goes and `{labels}` for the list of
options, and the labels (each with the words the teacher should answer with). This one comes from the
repository's examples.

In [ ]:
!curl -sLO https://raw.githubusercontent.com/Andreas-Hum/tinydistill/main/examples/financial-tweets/task.yaml
print(open("task.yaml").read())

## 2. Label, train, export and evaluate

`tinydistill run` does every step: the teacher scores each label for each training text (one forward pass per
text, no text generation), the student trains on those soft labels, the student is exported to ONNX (and
int8), and both are evaluated on the held-out test split, which here has gold labels.

`--val-share 0.05` holds out 5% of the training texts and keeps the training epoch that does best on them.

In [ ]:
!tinydistill run task.yaml \
  --data hf:zeroshot/twitter-financial-news-sentiment:train --limit 2000 \
  --test hf:zeroshot/twitter-financial-news-sentiment:validation --test-limit 500 \
  --teacher Qwen/Qwen2.5-1.5B-Instruct --val-share 0.05 --int8 --out run

## 3. Results

Accuracy of the teacher and the student on the test texts, their agreement, latency and size.

In [ ]:
from IPython.display import Markdown

Markdown(open("run/results.md").read())

## 4. What the student learned

**Soft labels.** The teacher does not give the student one answer per text but a probability for every label.
The student learns to reproduce that whole distribution, including how unsure the teacher is.

In [ ]:
from tinydistill import plots

plots.soft_labels("run");

**Accuracy and speed.** The student runs on a CPU in about a millisecond; the teacher needs a GPU and much
longer. The dashed line is the same student trained on all gold labels (here there is no such run, so it is
absent).

In [ ]:
plots.speed_accuracy("run");

**Training.** The loss on the training texts and on the held-out texts, per epoch; the dashed line marks the
epoch that was kept.

In [ ]:
plots.training("run");

**Where it goes wrong.** F1 per label for teacher and student, and the student's most common confusions.

In [ ]:
plots.per_label("run")
plots.confusions("run");

**Cascade.** The student can pass the texts it is least sure about to the teacher: more accuracy for more
latency. This curve shows the trade-off on the test texts.

In [ ]:
plots.cascade("run");

## 5. Use the student

The exported model needs only `onnxruntime` and a tokenizer, no PyTorch.

In [ ]:
from tinydistill import Predictor

clf = Predictor("run/onnx")  # uses the int8 model
texts = [
    "Shares jump 12% after the company beats earnings expectations",
    "Regulators open an investigation into the bank's lending practices",
    "The board will meet on Thursday to discuss the annual report",
]
for text, result in zip(texts, clf.describe(texts), strict=True):
    print(f"{result['label']:>8}  {result['confidence']:.2f}  {text}")

## 6. Where next

- **A stronger teacher** helps most: try `--teacher Qwen/Qwen2.5-7B-Instruct --load-4bit` (needs the `4bit`
  extra: `pip install "tinydistill[hf,4bit,plot]"`).
- **Your own task:** the [your own task notebook](https://colab.research.google.com/github/Andreas-Hum/tinydistill/blob/main/notebooks/your_task.ipynb) writes a task file for your labels
  and runs on your texts.
- **Deploy:** the [deploy notebook](https://colab.research.google.com/github/Andreas-Hum/tinydistill/blob/main/notebooks/deploy.ipynb) shows the server, the browser page and the Hugging
  Face Hub.
- The [README](https://github.com/Andreas-Hum/tinydistill) has the full documentation and benchmarks.